In [17]:
"""
MIND + Chroma RAG-recommendation pipeline with SAE-based gender-representation probing.

Gender tag source: Dacon & Liu (2021) -- majority gendered-word count per article,
supplied as a dict: article_id -> 'M'/'F' (binary, only articles with a clear majority).

Pipeline:
  1. Pull article embeddings + text from Chroma (paginated, avoids SQLite variable limit)
  2. Merge in the external gender_map dict -> article_df with 'gender_tag' column
  3. Train a TopK SAE on article embeddings; item-level probe for gender-correlated latents
  4. Build user click histories from the MIND behaviors file
  5. Build user profile embeddings from history text; retrieve top-K articles per user
  6. Calibration gap, TWO versions:
       a) history-based: retrieved F-ratio vs. historical (browsed) F-ratio
       b) click-based: retrieved F-ratio vs. click-through-revealed F-ratio (from Impressions),
          plus an aggregate CTR-by-gender check that controls for exposure imbalance

Adjust CONFIG and the two "ADAPT THIS" data-loading sections to match your setup.
"""

import ast
import json
import numpy as np
import pandas as pd
import pickle
import re
import torch
import torch.nn as nn
from tqdm import tqdm
from scipy.stats import pointbiserialr, wilcoxon
from statsmodels.stats.multitest import multipletests

# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
CHROMA_PATH = "/Users/jessicakahn/Documents/repos/MIND/mind_chroma_db"
CHROMA_COLLECTION_NAME = "mind_news"
CHROMA_BATCH_SIZE = 500

MAX_PROFILE_ITEMS = 20        # most recent N history articles per user profile
TOP_K_RETRIEVE = 10

SAE_K = 32
SAE_EXPANSION = 8
SAE_EPOCHS = 200
SAE_BATCH_SIZE = 256
SAE_LR = 1e-3
SEED = 42
GENDERED_WORDS_PATH = "/Users/jessicakahn/Documents/repos/MIND/data/gender_words.json"
with open(GENDERED_WORDS_PATH, 'r', encoding='utf-8') as file:
    gender_dict = json.load(file)
GENDERED_WORDS = list(gender_dict['male']) + list(gender_dict['female'])

# ------------------------------------------------------------------
# 1. Load from Chroma (paginated)
# ------------------------------------------------------------------
def load_mind_from_chroma(collection, batch_size=CHROMA_BATCH_SIZE):
    total = collection.count()
    ids, texts, embeddings = [], [], []
    for offset in range(0, total, batch_size):
        result = collection.get(limit=batch_size, offset=offset,
                                 include=['embeddings', 'documents'])
        ids.extend(result['ids'])
        texts.extend(result['documents'])
        embeddings.extend(result['embeddings'])
    return ids, texts, np.array(embeddings)


# ------------------------------------------------------------------
# 2. Merge in the external gender_map dict
# ------------------------------------------------------------------
def build_article_df(ids, texts, gender_map):
    """
    gender_map: dict article_id -> 'M'/'F' (adjust label_map below if yours uses 0/1 instead)
    """
    df = pd.DataFrame({'item_id': ids, 'text': texts})
    df['gender_tag'] = df['item_id'].map(gender_map)
    df['embedding_idx'] = np.arange(len(df))

    n_missing = df['gender_tag'].isna().sum()
    print(f"{n_missing} / {len(df)} articles have no gender tag (dropped from gender analyses)")
    return df


# ------------------------------------------------------------------
# 3. TopK SAE (reference implementation -- swap for your own class if it differs)
# ------------------------------------------------------------------
class TopKSAE(nn.Module):
    def __init__(self, d_in, d_latent, k):
        super().__init__()
        self.k = k
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.encoder = nn.Linear(d_in, d_latent, bias=True)
        self.decoder = nn.Linear(d_latent, d_in, bias=False)
        with torch.no_grad():
            self.decoder.weight.data = self.encoder.weight.data.t().clone()
        self._normalize_decoder()

    def _normalize_decoder(self):
        with torch.no_grad():
            norms = self.decoder.weight.norm(dim=0, keepdim=True).clamp_min(1e-8)
            self.decoder.weight.div_(norms)

    def encode(self, x):
        pre_act = torch.relu(self.encoder(x - self.b_dec))
        topk_vals, topk_idx = torch.topk(pre_act, self.k, dim=-1)
        latent = torch.zeros_like(pre_act)
        latent.scatter_(-1, topk_idx, topk_vals)
        return latent

    def decode(self, latent):
        return self.decoder(latent) + self.b_dec

    def forward(self, x):
        latent = self.encode(x)
        return self.decode(latent), latent


def train_sae(X, d_latent_mult=SAE_EXPANSION, k=SAE_K, epochs=SAE_EPOCHS,
              batch_size=SAE_BATCH_SIZE, lr=SAE_LR, seed=SEED):
    torch.manual_seed(seed)
    d_in = X.shape[1]
    sae = TopKSAE(d_in, d_in * d_latent_mult, k=k)
    opt = torch.optim.Adam(sae.parameters(), lr=lr)

    for epoch in range(epochs):
        perm = torch.randperm(X.shape[0])
        total_recon, n_batches = 0.0, 0
        fired_any = None

        for i in range(0, X.shape[0], batch_size):
            batch = X[perm[i:i + batch_size]]
            recon, latent = sae(batch)
            loss = ((recon - batch) ** 2).mean()

            opt.zero_grad()
            loss.backward()
            opt.step()
            sae._normalize_decoder()

            total_recon += loss.item()
            n_batches += 1
            fired = (latent != 0).any(dim=0)
            fired_any = fired if fired_any is None else (fired_any | fired)

        if epoch % 20 == 0 or epoch == epochs - 1:
            dead_frac = 1 - fired_any.float().mean().item()
            print(f"epoch {epoch}: recon={total_recon / n_batches:.6f}  dead_frac={dead_frac:.3f}")

    return sae


def get_latents(sae, X, batch_size=1024):
    sae.eval()
    out = []
    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            _, latent = sae(X[i:i + batch_size])
            out.append(latent)
    return torch.cat(out).numpy()


# ------------------------------------------------------------------
# 4. Item-level gender probe
# ------------------------------------------------------------------
def item_gender_probe(item_latents, gender_tags):
    mask = gender_tags.isin(['M', 'F']).values
    latents_sub = item_latents[mask]
    gender_binary = (gender_tags[mask] == 'F').astype(int).values

    results = []
    for latent_idx in range(latents_sub.shape[1]):
        col = latents_sub[:, latent_idx]
        if col.std() == 0:
            continue
        r, p = pointbiserialr(gender_binary, col)
        results.append((latent_idx, r, p))

    probe = pd.DataFrame(results, columns=['latent_idx', 'r', 'p'])
    probe['p_adj'] = multipletests(probe['p'], method='fdr_bh')[1]
    return probe.sort_values('r', key=abs, ascending=False).reset_index(drop=True)


def top_examples_for_latent(latents, texts, latent_idx, top_n=10):
    col = latents[:, latent_idx]
    idx = np.argsort(-col)[:top_n]
    return [(texts[i], float(col[i])) for i in idx]


# ------------------------------------------------------------------
# 5. User histories from the MIND behaviors file
# ------------------------------------------------------------------
def build_user_histories(behaviors_df, history_col='History'):
    # def parse_history(h):
        # return ast.literal_eval(h) if isinstance(h, str) else h

    behaviors_df = behaviors_df.copy()
    tqdm.pandas(desc="Parsing history strings")
    behaviors_df['_history_parsed'] = behaviors_df[history_col].fillna('').apply(str.split)


    user_histories = {}
    grouped = behaviors_df.groupby('UserID')
    for user_id, group in tqdm(grouped, desc="Building user histories", total=grouped.ngroups):
        # preserve order, de-duplicate without using a set (dict.fromkeys keeps first-seen order
        # and is insertion-ordered, unlike set -- no hash-randomization issue)
        ordered_ids = []
        for hist in group['_history_parsed']:
            ordered_ids.extend(hist)
        deduped_ordered = list(dict.fromkeys(ordered_ids))  # dedupe, preserve order, no set() involved
        user_histories[user_id] = deduped_ordered
    return user_histories


# ------------------------------------------------------------------
# 5b. User impressions (click-through signal) from the MIND behaviors file
# ------------------------------------------------------------------
def parse_impressions(impression_str):
    """'N50014-0 N23877-0 N49712-1' -> [('N50014', 0), ('N23877', 0), ('N49712', 1)]"""
    pairs = []
    for token in impression_str.split():
        item_id, label = token.rsplit('-', 1)
        pairs.append((item_id, int(label)))
    return pairs


def build_user_impressions(behaviors_df, impressions_col='Impressions'):
    behaviors_df = behaviors_df.copy()
    behaviors_df['_impressions_parsed'] = behaviors_df[impressions_col].apply(parse_impressions)

    user_impressions = {}
    for user_id, group in behaviors_df.groupby('UserID'):
        all_pairs = []
        for pairs in group['_impressions_parsed']:
            all_pairs.extend(pairs)
        user_impressions[user_id] = all_pairs  # list of (item_id, label)
    return user_impressions


# ------------------------------------------------------------------
# 6. Profile text + embeddings + retrieval
# ------------------------------------------------------------------
def build_user_profile_text(user_id, history_ids, item_id_to_text, max_items=MAX_PROFILE_ITEMS):
    valid_ids = [i for i in history_ids if i in item_id_to_text.index][-max_items:]
    texts = [item_id_to_text.loc[i] for i in valid_ids]
    return "; ".join(texts)


def retrieve_top_k_ids(user_embeddings, item_embeddings, item_ids_array, k=TOP_K_RETRIEVE, batch_size=500):
    user_norm = user_embeddings / (np.linalg.norm(user_embeddings, axis=1, keepdims=True) + 1e-8)
    item_norm = item_embeddings / (np.linalg.norm(item_embeddings, axis=1, keepdims=True) + 1e-8)

    n_users = user_norm.shape[0]
    all_top_k_ids = np.empty((n_users, k), dtype=object)

    for start in range(0, n_users, batch_size):
        end = min(start + batch_size, n_users)
        batch_sims = user_norm[start:end] @ item_norm.T   # (batch_size, n_items) -- much smaller
        top_k_idx = np.argsort(-batch_sims, axis=1)[:, :k]
        all_top_k_ids[start:end] = item_ids_array[top_k_idx]

        if start % 5000 == 0:
            print(f"  retrieved for {start}/{n_users} users")

    return all_top_k_ids


# ------------------------------------------------------------------
# 7a. Calibration gap -- history-based
# ------------------------------------------------------------------
def gender_ratio(item_ids, article_df, include_neutral=True):
    if include_neutral:
        sub = article_df[article_df.item_id.isin(item_ids) & article_df.gender_tag.isin(['M', 'F', 'Neutral'])]
    else:
        sub = article_df[article_df.item_id.isin(item_ids) & article_df.gender_tag.isin(['M', 'F'])]
    return (sub.gender_tag == 'F').mean() if len(sub) > 0 else np.nan


def calibration_gap_history(user_histories, user_retrievals, article_df):
    rows = []
    for user_id, hist_ids in user_histories.items():
        retrieved_ids = user_retrievals.get(user_id, [])
        rows.append({
            'user_id': user_id,
            'hist_ratio_F': gender_ratio(hist_ids, article_df),
            'retrieved_ratio_F': gender_ratio(retrieved_ids, article_df),
        })
    df = pd.DataFrame(rows).dropna()
    df['gap'] = df['retrieved_ratio_F'] - df['hist_ratio_F']
    return df


# ------------------------------------------------------------------
# 7b. Calibration gap -- click-based (revealed preference from Impressions)
# ------------------------------------------------------------------
def calibration_gap_clicks(user_impressions, user_retrievals, article_df):
    """
    click_ratio_F: among articles the user actually CLICKED (label=1) when shown,
                   fraction that were F-tagged -- this is revealed preference,
                   distinct from browsing history which includes non-impression clicks.
    """
    rows = []
    for user_id, pairs in user_impressions.items():
        clicked_ids = [item_id for item_id, label in pairs if label == 1]
        retrieved_ids = user_retrievals.get(user_id, [])
        rows.append({
            'user_id': user_id,
            'click_ratio_F': gender_ratio(clicked_ids, article_df),
            'retrieved_ratio_F': gender_ratio(retrieved_ids, article_df),
        })
    df = pd.DataFrame(rows).dropna()
    df['gap'] = df['retrieved_ratio_F'] - df['click_ratio_F']
    return df


def ctr_by_gender(user_impressions, article_df):
    """
    Aggregate click-through rate for M/F/Neutral-tagged shown articles, across ALL users.
    This controls for exposure imbalance: if a group is shown less often,
    a raw click-count comparison would be misleading -- CTR normalizes for that.
    Per-user CTR-by-gender is usually too sparse (few impressions per user) to be meaningful,
    so this is computed in aggregate.
    """
    shown = {'M': 0, 'F': 0, 'Neutral': 0}
    clicked = {'M': 0, 'F': 0, 'Neutral': 0}
    item_gender = article_df.set_index('item_id')['gender_tag']

    for pairs in tqdm(user_impressions.values(), desc="Computing CTR by gender", total=len(user_impressions)):
        for item_id, label in pairs:
            tag = item_gender.get(item_id, None)
            if tag in shown:
                shown[tag] += 1
                clicked[tag] += label

    ctr = {tag: (clicked[tag] / shown[tag] if shown[tag] > 0 else np.nan) for tag in shown}

    print(f"Shown    -- M: {shown['M']}, F: {shown['F']}, Neutral: {shown['Neutral']}")
    print(f"Clicked  -- M: {clicked['M']}, F: {clicked['F']}, Neutral: {clicked['Neutral']}")
    print(f"CTR      -- M: {ctr['M']:.4f}, F: {ctr['F']:.4f}, Neutral: {ctr['Neutral']:.4f}")

    return {
        'ctr_M': ctr['M'], 'ctr_F': ctr['F'], 'ctr_Neutral': ctr['Neutral'],
        'shown_M': shown['M'], 'shown_F': shown['F'], 'shown_Neutral': shown['Neutral'],
        'clicked_M': clicked['M'], 'clicked_F': clicked['F'], 'clicked_Neutral': clicked['Neutral'],
    }

def describe_gender_distribution(article_df, user_histories=None, user_retrievals=None, user_impressions=None):
    """
    Prints and returns full M/F/Neutral descriptive stats at each stage available.
    Pass whichever of user_histories/user_retrievals/user_impressions you have -- all optional.
    """
    results = {}

    # --- Corpus-level ---
    print("=== Corpus (all articles) ===")
    corpus_counts = article_df['gender_tag'].value_counts(dropna=False)
    corpus_props = article_df['gender_tag'].value_counts(dropna=False, normalize=True)
    corpus_stats = pd.DataFrame({'count': corpus_counts, 'proportion': corpus_props})
    print(corpus_stats)
    results['corpus'] = corpus_stats

    # --- History-level (pooled across all users' histories) ---
    if user_histories is not None:
        print("\n=== User histories (pooled across all users) ===")
        all_hist_ids = [i for ids in user_histories.values() for i in ids]
        hist_tags = article_df.set_index('item_id')['gender_tag'].reindex(all_hist_ids)
        hist_counts = hist_tags.value_counts(dropna=False)
        hist_props = hist_tags.value_counts(dropna=False, normalize=True)
        hist_stats = pd.DataFrame({'count': hist_counts, 'proportion': hist_props})
        print(hist_stats)
        results['history'] = hist_stats

    # --- Retrieval-level (pooled across all users' retrieved sets) ---
    if user_retrievals is not None:
        print("\n=== Retrieved articles (pooled across all users) ===")
        all_retrieved_ids = [i for ids in user_retrievals.values() for i in ids]
        retr_tags = article_df.set_index('item_id')['gender_tag'].reindex(all_retrieved_ids)
        retr_counts = retr_tags.value_counts(dropna=False)
        retr_props = retr_tags.value_counts(dropna=False, normalize=True)
        retr_stats = pd.DataFrame({'count': retr_counts, 'proportion': retr_props})
        print(retr_stats)
        results['retrieved'] = retr_stats

    # --- Impressions: shown vs. clicked breakdown ---
    if user_impressions is not None:
        print("\n=== Impressions: SHOWN (pooled across all users) ===")
        all_shown_ids = [item_id for pairs in user_impressions.values() for item_id, _ in pairs]
        shown_tags = article_df.set_index('item_id')['gender_tag'].reindex(all_shown_ids)
        shown_counts = shown_tags.value_counts(dropna=False)
        shown_props = shown_tags.value_counts(dropna=False, normalize=True)
        shown_stats = pd.DataFrame({'count': shown_counts, 'proportion': shown_props})
        print(shown_stats)
        results['shown'] = shown_stats

        print("\n=== Impressions: CLICKED (pooled across all users) ===")
        all_clicked_ids = [item_id for pairs in user_impressions.values() for item_id, label in pairs if label == 1]
        clicked_tags = article_df.set_index('item_id')['gender_tag'].reindex(all_clicked_ids)
        clicked_counts = clicked_tags.value_counts(dropna=False)
        clicked_props = clicked_tags.value_counts(dropna=False, normalize=True)
        clicked_stats = pd.DataFrame({'count': clicked_counts, 'proportion': clicked_props})
        print(clicked_stats)
        results['clicked'] = clicked_stats

    return results

def summarize_calibration(cal_df, hist_or_click_col):
    print(f"n users: {len(cal_df)}")
    print(f"mean {hist_or_click_col}: {cal_df[hist_or_click_col].mean():.3f}")
    print(f"mean retrieved_ratio_F: {cal_df['retrieved_ratio_F'].mean():.3f}")
    print(f"mean gap (retrieved - {hist_or_click_col}): {cal_df['gap'].mean():.4f}")
    print(f"fraction of users where retrieval REDUCES F-representation: {(cal_df['gap'] < 0).mean():.3f}")
    stat, p = wilcoxon(cal_df['retrieved_ratio_F'], cal_df[hist_or_click_col])
    print(f"Wilcoxon signed-rank (retrieved vs {hist_or_click_col}): stat={stat:.1f}, p={p:.4g}")


# Build a single regex with word boundaries, case-insensitive
_pattern = re.compile(r'\b(' + '|'.join(re.escape(w) for w in GENDERED_WORDS) + r')\b', flags=re.IGNORECASE)

def strip_gendered_words(text):
    return _pattern.sub('', text)


def build_degendered_texts(article_df):
    """
    Returns a new list of texts with gendered words stripped, aligned to article_df's row order.
    IMPORTANT: gender_tag stays based on the ORIGINAL text (that's the ground-truth label from
    Dacon & Liu's method) -- only the text going INTO the embedder changes.
    """
    return article_df['text'].apply(strip_gendered_words).tolist()

# ------------------------------------------------------------------
# Main
# ------------------------------------------------------------------


In [19]:
if __name__ == "__main__":
    # --- ADAPT THIS: wire up your actual Chroma client, gender_map, behaviors_df, embed_model ---
    import chromadb
    from sentence_transformers import SentenceTransformer

    client = chromadb.PersistentClient(path=CHROMA_PATH)
    collection = client.get_collection(name=CHROMA_COLLECTION_NAME)

    with open("/Users/jessicakahn/Documents/repos/MIND/data/gender_dict.pkl", "rb") as file:
        gender_map = pickle.load(file)

    behaviors_df = pd.read_csv(
        '/Users/jessicakahn/Documents/repos/MIND/data/MINDsmall_train/behaviors.tsv', sep='\t', header=None,
        names=['ImpressionID', 'UserID', 'Time', 'History', 'Impressions']
    )

    embed_model = SentenceTransformer("all-MiniLM-L6-v2")  # match whatever encoder built your Chroma embeddings

    # results = main(collection, gender_map, behaviors_df, embed_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8196.04it/s]


Loading articles from Chroma...
0 / 51282 articles have no gender tag (dropped from gender analyses)
Re-embedding degendered text...


Batches: 100%|██████████| 1603/1603 [01:07<00:00, 23.79it/s]



Building user histories and impressions...


Building user histories: 100%|██████████| 50000/50000 [00:01<00:00, 32376.94it/s]



Building user profiles and embedding...
49108 users with valid profiles


Batches: 100%|██████████| 1535/1535 [03:17<00:00,  7.78it/s]



Retrieving top-K articles per user...
  retrieved for 0/49108 users
  retrieved for 5000/49108 users
  retrieved for 10000/49108 users
  retrieved for 15000/49108 users
  retrieved for 20000/49108 users
  retrieved for 25000/49108 users
  retrieved for 30000/49108 users
  retrieved for 35000/49108 users
  retrieved for 40000/49108 users
  retrieved for 45000/49108 users

=== Calibration gap: retrieval vs. HISTORY ===
n users: 49108
mean hist_ratio_F: 0.109
mean retrieved_ratio_F: 0.136
mean gap (retrieved - hist_ratio_F): 0.0273
fraction of users where retrieval REDUCES F-representation: 0.380
Wilcoxon signed-rank (retrieved vs hist_ratio_F): stat=287451631.0, p=1.234e-98

=== Calibration gap: retrieval vs. CLICK-THROUGH (revealed preference) ===
n users: 49108
mean click_ratio_F: 0.089
mean retrieved_ratio_F: 0.136
mean gap (retrieved - click_ratio_F): 0.0468
fraction of users where retrieval REDUCES F-representation: 0.212
Wilcoxon signed-rank (retrieved vs click_ratio_F): stat=1579

Computing CTR by gender: 100%|██████████| 50000/50000 [00:10<00:00, 4867.34it/s]


Shown    -- M: 829526, F: 420923, Neutral: 4592995
Clicked  -- M: 34422, F: 21850, Neutral: 180072
CTR      -- M: 0.0415, F: 0.0519, Neutral: 0.0392
{'ctr_M': 0.041495986864787844, 'ctr_F': 0.0519097317086498, 'ctr_Neutral': 0.039205790557141906, 'shown_M': 829526, 'shown_F': 420923, 'shown_Neutral': 4592995, 'clicked_M': 34422, 'clicked_F': 21850, 'clicked_Neutral': 180072}

=== Gender distribution diagnostics (M/F/Neutral) ===
=== Corpus (all articles) ===
            count  proportion
gender_tag                   
Neutral     37576    0.732733
M           10281    0.200480
F            3425    0.066788

=== User histories (pooled across all users) ===
             count  proportion
gender_tag                    
Neutral     639483    0.698880
M           175699    0.192018
F            99829    0.109101

=== Retrieved articles (pooled across all users) ===
             count  proportion
gender_tag                    
Neutral     316213    0.643913
M           108149    0.220227
F   

In [20]:
print("Loading articles from Chroma...")
ids, texts, embeddings = load_mind_from_chroma(collection)
article_df = build_article_df(ids, texts, gender_map)

Loading articles from Chroma...
0 / 51282 articles have no gender tag (dropped from gender analyses)


In [23]:
print("\nBuilding user histories and impressions...")
user_histories = build_user_histories(behaviors_df)
user_impressions = build_user_impressions(behaviors_df)

print("\nBuilding user profiles and embedding...")
item_id_to_text = article_df.set_index('item_id')['text']
user_ids = list(user_histories.keys())
profile_texts = [build_user_profile_text(uid, user_histories[uid], item_id_to_text) for uid in user_ids]

valid = [len(t) > 0 for t in profile_texts]
user_ids = [u for u, v in zip(user_ids, valid) if v]
profile_texts = [t for t, v in zip(profile_texts, valid) if v]
print(f"{len(user_ids)} users with valid profiles")

user_embeddings = embed_model.encode(profile_texts, convert_to_numpy=True, show_progress_bar=True)


Building user histories and impressions...


Building user histories: 100%|██████████| 50000/50000 [00:01<00:00, 31382.41it/s]



Building user profiles and embedding...
49108 users with valid profiles


Batches: 100%|██████████| 1535/1535 [03:35<00:00,  7.11it/s]


In [26]:
print("\nRetrieving top-K articles per user...")
item_ids_array = np.array(article_df['item_id'].astype(str).tolist(), dtype=object)
top_k_ids = retrieve_top_k_ids(user_embeddings, embeddings, item_ids_array)
user_retrievals = {uid: top_k_ids[i].tolist() for i, uid in enumerate(user_ids)}


Retrieving top-K articles per user...
  retrieved for 0/49108 users
  retrieved for 5000/49108 users
  retrieved for 10000/49108 users
  retrieved for 15000/49108 users
  retrieved for 20000/49108 users
  retrieved for 25000/49108 users
  retrieved for 30000/49108 users
  retrieved for 35000/49108 users
  retrieved for 40000/49108 users
  retrieved for 45000/49108 users


In [ ]:
# def main(chroma_collection, gender_map, behaviors_df, embed_model):


# degendered_texts = build_degendered_texts(article_df)
# print('Degendered text', degendered_texts[0:10])
# print("Re-embedding degendered text...")
# degendered_embeddings = embed_model.encode(degendered_texts, convert_to_numpy=True, show_progress_bar=True)








print("\n=== Calibration gap: retrieval vs. HISTORY ===")
user_histories_filtered = {uid: user_histories[uid] for uid in user_ids}
cal_hist = calibration_gap_history(user_histories_filtered, user_retrievals, article_df)
summarize_calibration(cal_hist, 'hist_ratio_F')

print("\n=== Calibration gap: retrieval vs. CLICK-THROUGH (revealed preference) ===")
user_impressions_filtered = {uid: user_impressions[uid] for uid in user_ids if uid in user_impressions}
cal_click = calibration_gap_clicks(user_impressions_filtered, user_retrievals, article_df)
summarize_calibration(cal_click, 'click_ratio_F')

print("\n=== CTR by gender (exposure-normalized, aggregate across all users) ===")
ctr_results = ctr_by_gender(user_impressions, article_df)
print(ctr_results)
print("\n=== Gender distribution diagnostics (M/F/Neutral) ===")
gender_stats = describe_gender_distribution(
        article_df,
        user_histories=user_histories_filtered,
        user_retrievals=user_retrievals,
        user_impressions=user_impressions_filtered,
    )
print(gender_stats)

# cal_hist.to_csv("calibration_gap_history.csv", index=False)
# cal_click.to_csv("calibration_gap_clicks.csv", index=False)
# item_probe.to_csv("item_gender_probe.csv", index=False)




In [21]:
from sklearn.metrics.pairwise import cosine_similarity

for tag in ['Neutral', 'M', 'F']:
    mask = (article_df['gender_tag'] == tag).values
    group_embeddings = embeddings[mask]
    centroid = group_embeddings.mean(axis=0, keepdims=True)
    sims_to_centroid = cosine_similarity(group_embeddings, centroid).flatten()
    print(f"{tag}: mean similarity to own-group centroid = {sims_to_centroid.mean():.3f}, "
          f"std = {sims_to_centroid.std():.3f}")

Neutral: mean similarity to own-group centroid = 0.282, std = 0.101
M: mean similarity to own-group centroid = 0.330, std = 0.098
F: mean similarity to own-group centroid = 0.360, std = 0.100


In [28]:
m_centroid = embeddings[(article_df['gender_tag']=='M').values].mean(axis=0, keepdims=True)
f_centroid = embeddings[(article_df['gender_tag']=='F').values].mean(axis=0, keepdims=True)
n_centroid = embeddings[(article_df['gender_tag']=='Neutral').values].mean(axis=0, keepdims=True)

sample_profiles = user_embeddings
sim_to_m = cosine_similarity(sample_profiles, m_centroid).flatten()
sim_to_f = cosine_similarity(sample_profiles, f_centroid).flatten()
sim_to_n = cosine_similarity(sample_profiles, n_centroid).flatten()

print(f"Mean profile similarity -- M centroid: {sim_to_m.mean():.3f}, "
      f"F centroid: {sim_to_f.mean():.3f}, Neutral centroid: {sim_to_n.mean():.3f}")

Mean profile similarity -- M centroid: 0.335, F centroid: 0.336, Neutral centroid: 0.340


In [27]:
all_retrieved_flat = [item_id for uid in user_ids for item_id in user_retrievals[uid]]
retrieved_series = pd.Series(all_retrieved_flat)
retrieved_tags = article_df.set_index('item_id')['gender_tag'].reindex(retrieved_series)

for tag in ['Neutral', 'M', 'F']:
    tag_mask = retrieved_tags == tag
    total_slots = tag_mask.sum()
    distinct_items = retrieved_series[tag_mask.values].nunique()
    print(f"{tag}: {total_slots} total retrieval slots filled by {distinct_items} distinct articles "
          f"(avg {total_slots/distinct_items:.1f} retrievals per article)")

Neutral: 316213 total retrieval slots filled by 25165 distinct articles (avg 12.6 retrievals per article)
M: 108149 total retrieval slots filled by 7368 distinct articles (avg 14.7 retrievals per article)
F: 66718 total retrieval slots filled by 2840 distinct articles (avg 23.5 retrievals per article)


In [29]:
#  Debug - delete
user_df_pickle_path = '/Users/jessicakahn/Documents/repos/MIND/data/retrieved_user.pkl'
user_df = pd.read_pickle(user_df_pickle_path)

In [33]:
user_df.head()

,ImpressionID,UserID,Time,History,Impressions,history_list,history_len,hist_,history_gender,records,averaged_embed,retrieved,retrieved_gender
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...,"[N31739, N6072, N63045, N23979, N35656, N43353...",82.0,"[N48390, N58224, N48742, N35458, N24611, N3750...","[Neutral, Neutral, M, Neutral, F, M, Neutral, ...","{'ids': ['N35458', 'N37509', 'N48742', 'N24611...","[-0.022237273445352913, 0.06586747923865914, -...","{'ids': [['N43585', 'N25785', 'N19041', 'N3188...","[Neutral, Neutral, Neutral, M, Neutral, Neutra..."
2,3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...,"[N10732, N25792, N7563, N21087, N41087, N5445,...",16.0,"[N60384, N46616, N52500, N33164, N47289, N2423...","[Neutral, Neutral, Neutral, Neutral, Neutral, ...","{'ids': ['N24233', 'N49475', 'N46616', 'N52500...","[0.017427177540957926, 0.031492797564715146, 0...","{'ids': [['N52500', 'N18870', 'N24233', 'N4139...","[Neutral, Neutral, Neutral, Neutral, Neutral, ..."
3,4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0,"[N45729, N2203, N871, N53880, N41375, N43142, ...",10.0,"[N45729, N2203, N871, N53880, N41375, N43142, ...","[Neutral, Neutral, M, Neutral, Neutral, Neutra...","{'ids': ['N53880', 'N45729', 'N33013', 'N43142...","[-0.026858700846787542, 0.013809896679595113, ...","{'ids': [['N2203', 'N3149', 'N36352', 'N46011'...","[Neutral, Neutral, Neutral, Neutral, Neutral, ..."
5,6,U19739,11/11/2019 6:52:13 PM,N39074 N14343 N32607 N32320 N22007 N442 N19001...,N21119-1 N53696-0 N33619-1 N25722-0 N2869-0,"[N39074, N14343, N32607, N32320, N22007, N442,...",36.0,"[N42512, N58521, N62846, N14385, N47020, N2142...","[M, Neutral, Neutral, Neutral, Neutral, Neutra...","{'ids': ['N42512', 'N62846', 'N58521', 'N52121...","[-0.023250294021434255, 0.03033571145755963, 0...","{'ids': [['N17511', 'N62103', 'N39650', 'N5852...","[Neutral, Neutral, Neutral, Neutral, Neutral, ..."
6,7,U8355,11/11/2019 12:22:09 PM,N8419 N15771 N1431 N5888 N18663 N24123 N22130 ...,N51346-0 N33848-0 N15132-0 N10688-0 N6342-0 N6...,"[N8419, N15771, N1431, N5888, N18663, N24123, ...",35.0,"[N306, N30160, N41797, N47482, N2606, N20886, ...","[M, Neutral, Neutral, M, F, M, Neutral, Neutra...","{'ids': ['N47482', 'N306', 'N20886', 'N41797',...","[-0.014741261675953864, 0.053519477136433125, ...","{'ids': [['N2606', 'N47482', 'N40410', 'N1854'...","[F, M, Neutral, Neutral, Neutral, Neutral, Neu..."


In [32]:
user_df.iloc[0]

ImpressionID                                                        2
UserID                                                         U91836
Time                                            11/12/2019 6:11:30 PM
History             N31739 N6072 N63045 N23979 N35656 N43353 N8129...
Impressions         N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
history_list        [N31739, N6072, N63045, N23979, N35656, N43353...
history_len                                                      82.0
hist_               [N48390, N58224, N48742, N35458, N24611, N3750...
history_gender      [Neutral, Neutral, M, Neutral, F, M, Neutral, ...
records             {'ids': ['N35458', 'N37509', 'N48742', 'N24611...
averaged_embed      [-0.022237273445352913, 0.06586747923865914, -...
retrieved           {'ids': [['N43585', 'N25785', 'N19041', 'N3188...
retrieved_gender    [Neutral, Neutral, Neutral, M, Neutral, Neutra...
Name: 1, dtype: object

### Parse out retrieved re-ranked lists from Ollama

In [ ]:
news_df = pd.read_csv('/Users/jessicakahn/Documents/repos/MIND/data/MINDsmall_train/news.tsv',sep='\t',header=None)
news_df.columns = ['NewsID', 'Category', 'SubCategory', 'Title', 'Abstract', 'URL', 'TitleEntities', 'AbstractEntities']
import json

# assuming you have this from earlier
news_lookup = news_df.set_index("NewsID")[["Title", "Abstract"]].to_dict("index")
# news_df = news_df.set_index("NewsID")  # or whatever the actual ID column is called

In [180]:
# Load generated output
n=500
input_type='new_ids'
model="llama3.2"
with open(f"/Users/jessicakahn/Documents/repos/MIND/data/responses_{model}_{n}_{input_type}.json", "r") as f:
    results = json.load(f)  # this gives you the real full dict, correctly parsed

In [230]:
# results
news_df['NewsID'].unique()

<ArrowStringArray>
['N55528', 'N19639', 'N61837', 'N53526', 'N38324',  'N2073', 'N49186',
 'N59295', 'N24510', 'N39237',
 ...
  'N5027', 'N64760', 'N43432', 'N17258', 'N23858', 'N16909', 'N47585',
  'N7482', 'N34418', 'N44276']
Length: 51282, dtype: str

In [231]:
def parse_ranking(raw_output, expected_ids):
    """
    Extract article IDs (Nxxxxx pattern) directly via regex,
    robust to preamble text, truncation, malformed JSON, and duplicates.
    """
    # find all quoted or bare N-prefixed IDs
    ids = re.findall(r'N\d+', raw_output)
    
    # dedupe while preserving first-seen order
    seen = set()
    deduped = []
    for aid in ids:
        if aid in expected_ids and aid not in seen:  # only accept IDs that were actually candidates
            seen.add(aid)
            deduped.append(aid)
        if len(deduped) == len(expected_ids):  # stop once we have exactly the expected count
            break
    return deduped

def parse_ranking_with_diagnostics(raw_output, expected_ids):
    ids = re.findall(r'N\d+', raw_output)
    seen = set()
    deduped = []
    for aid in ids:
        if aid not in seen:
            seen.add(aid)
            deduped.append(aid)
    
    coverage = len(deduped) / len(expected_ids) if expected_ids else 0
    truncated = coverage < 0.9  # flag if we got less than 90% of expected candidates
    
    return {
        "ranked_ids": deduped,
        "coverage": coverage,
        "likely_truncated": truncated,
    }

def map_ranking_to_titles(ranked_ids, news_lookup):
    """Map article IDs to titles, flagging any missing ones."""
    return [
        {
            "rank": i + 1,
            "article_id": aid,
            "title": news_lookup.get(aid, {}).get("Title", "MISSING")
        }
        for i, aid in enumerate(ranked_ids)
    ]

parsed_results = {}
for user_impression_id, raw_output in results.items():
    ranked_ids = parse_ranking(raw_output, news_df['NewsID'].unique())
    parsed_results[user_impression_id] = map_ranking_to_titles(ranked_ids, news_lookup)


In [183]:
# parsed_results

In [79]:
# 'U87803:139007'
# behaviors_df[(behaviors_df['UserID']=='U87803')&(behaviors_df['ImpressionID']==139007)]['Impressions'].values

In [80]:
# news_df[news_df['NewsID']=='N46466']

### Measure accuracy of retrieval

In [81]:
def parse_impressions(impressions_str):
    """
    Parse 'N123-0 N234-1 N345-0' into a dict {article_id: relevance_label}
    """
    pairs = impressions_str.strip().split()
    return {
        p.split("-")[0]: int(p.split("-")[1])
        for p in pairs
    }

# example
impressions = "N12345-0 N23456-1 N34567-0 N45678-0"
relevance = parse_impressions(impressions)
# {"N12345": 0, "N23456": 1, "N34567": 0, "N45678": 0}

In [100]:
def get_relevance_in_ranked_order(ranked_ids, relevance_dict):
    """
    Returns a list of 0/1 labels, ordered according to ranked_ids.
    Drops any IDs not found in relevance_dict (shouldn't happen, but be safe).
    """
    return [relevance_dict[aid] for aid in ranked_ids if aid in relevance_dict]

In [101]:
import numpy as np
from sklearn.metrics import roc_auc_score

def dcg_at_k(relevance, k):
    relevance = np.asarray(relevance)[:k]
    if relevance.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, relevance.size + 2))
    return np.sum(relevance / discounts)

def ndcg_at_k(relevance, k):
    ideal_relevance = sorted(relevance, reverse=True)
    idcg = dcg_at_k(ideal_relevance, k)
    if idcg == 0:
        return 0.0
    return dcg_at_k(relevance, k) / idcg

def mrr(relevance):
    for i, rel in enumerate(relevance, start=1):
        if rel == 1:
            return 1.0 / i
    return 0.0

def auc(relevance):
    # AUC requires both classes present; skip impressions with only clicks or only non-clicks
    if len(set(relevance)) < 2:
        return None
    scores = list(range(len(relevance), 0, -1))  # rank position as pseudo-score, higher = better
    return roc_auc_score(relevance, scores)

def evaluate_ranking(relevance):
    return {
        "auc": auc(relevance),
        "mrr": mrr(relevance),
        "ndcg@5": ndcg_at_k(relevance, 5),
        "ndcg@10": ndcg_at_k(relevance, 10),
    }

In [102]:
# parsed_results
behaviors_df['user_impression_id'] = behaviors_df['UserID']+':'+behaviors_df['ImpressionID'].astype(str)

In [103]:
results_rows = []

for user_impression_id, ranked_ids in parsed_results.items():
    # ranked_ids should be your list of article_ids in LLM-ranked order
    ranked_id_list = [entry["article_id"] for entry in ranked_ids]  # if using the dict format from earlier
    
    impressions_str = behaviors_df.loc[
        behaviors_df["user_impression_id"] == user_impression_id, "Impressions"
    ].values[0]
    
    relevance_dict = parse_impressions(impressions_str)
    relevance_ordered = get_relevance_in_ranked_order(ranked_id_list, relevance_dict)
    
    metrics = evaluate_ranking(relevance_ordered)
    metrics["impression_id"] = user_impression_id
    results_rows.append(metrics)

metrics_df = pd.DataFrame(results_rows)

# aggregate — this is what you'd report
summary = metrics_df[["auc", "mrr", "ndcg@5", "ndcg@10"]].mean()
print(summary)

auc        1.0000
mrr        0.0046
ndcg@5     0.0046
ndcg@10    0.0046
dtype: float64


In [97]:
# metrics_df

In [226]:
# # grab one impression to debug
# sample_id = "U27020:137620"

# ranked_id_list = [entry["article_id"] for entry in parsed_results[sample_id]]
# print("Ranked IDs:", ranked_id_list[:5])

# impressions_str = behaviors_df.loc[
#     behaviors_df["user_impression_id"] == sample_id, "Impressions"
# ].values

# print("Impressions lookup result:", impressions_str)  # <-- check this isn't empty first

In [95]:
# parsed_results

In [105]:
# for open retrieval
def recall_at_k(retrieved_ids, relevance_dict, k=10):
    clicked = {aid for aid, label in relevance_dict.items() if label == 1}
    retrieved_top_k = set(retrieved_ids[:k])
    return len(clicked & retrieved_top_k) / len(clicked) if clicked else None

recall_results = []

for user_impression_id, ranked_entries in parsed_results.items():
    retrieved_ids = [entry["article_id"] for entry in ranked_entries]  # your open-retrieved IDs, in order

    row = behaviors_df.loc[behaviors_df["user_impression_id"] == user_impression_id]
    if row.empty:
        continue

    relevance_dict = parse_impressions(row["Impressions"].values[0])

    score = recall_at_k(retrieved_ids, relevance_dict, k=10)
    recall_results.append({"impression_id": user_impression_id, "recall@10": score})

recall_df = pd.DataFrame(recall_results)
print(recall_df["recall@10"].dropna().mean())

0.0018033333333333332


In [107]:
parsed_results

{'U41010:92845': [{'rank': 1,
   'article_id': 'N1743',
   'title': 'Could Kentucky, a deep red state, elect a Democrat as its next governor?'},
  {'rank': 2,
   'article_id': 'N3504',
   'title': 'Kentucky voters look to settle gubernatorial grudge match'},
  {'rank': 3,
   'article_id': 'N24874',
   'title': 'Gov. Matt Bevin concedes minutes before recanvass concludes'},
  {'rank': 4,
   'article_id': 'N58151',
   'title': 'Suburban rejection of Trump boosts Kentucky Democratic victory'},
  {'rank': 5,
   'article_id': 'N35615',
   'title': 'Group lists concerns about Kentucky gubernatorial election results'},
  {'rank': 6,
   'article_id': 'N20017',
   'title': "Analysis: Trump's GOP has no answer for suburban slide"},
  {'rank': 7,
   'article_id': 'N63719',
   'title': 'LIVE: Gov. Bevin holds press conference as recanvass wraps up in Kentucky'},
  {'rank': 8,
   'article_id': 'N43827',
   'title': "Beshear vs. Bevin: Kentucky governor's race could be decided by state legislature"}

Issue is that recall really only works for Impressions, not from open retrieval, so I need a separate measure for open retrieval quality?

In [ ]:
# Compare gender ranking of pre-reranked (retrieved) items to post-reranked items
# gender_map

In [113]:
# retrieved
ret_path = "/Users/jessicakahn/Documents/repos/MIND/data/output_user_retrievals.json"
with open(ret_path,'rb') as file:
    retrieved = json.load(file)


In [116]:
retrieved_gender = {k:[gender_map[i] for i in v] for k,v in retrieved.items()}

In [232]:
# remove impression id from re-ranked and dedupe - it probably doubles up the same retrievals
parsed_results_dedupe = {}
for k, v in parsed_results.items():
    user_id = k.split(':')[0]
    if user_id not in parsed_results_dedupe:
        article_list = [i['article_id'] for i in v]
        parsed_results_dedupe[user_id] = article_list
        


In [233]:
len(parsed_results),len(parsed_results_dedupe)

(500, 494)

In [191]:
# parsed_results_dedupe

In [234]:
# re-ranked
# /Users/jessicakahn/Documents/repos/MIND/data/responses_llama3_5000_new_ids.json
# ['article_id']
reranked_gender = {k:[gender_map.get(i) for i in v] for k,v in parsed_results_dedupe.items()}
# reranked_gender

In [201]:
# gender_map['N5503
# parsed_results_dedupe
# retrieved_rank

In [235]:
def build_rank_comparison(retrieved_dict, reranked_dict, gender_lookup):
    rows = []
    for user_id in reranked_dict:
        retrieved_rank = {aid: i+1 for i, aid in enumerate(retrieved_dict[user_id])}
        reranked_rank = {aid: i+1 for i, aid in enumerate(reranked_dict[user_id])}
        # print(retrieved_rank, reranked_rank)
        
        for aid in retrieved_rank:  # same set of IDs in both, per your setup
            rows.append({
                "user_id": user_id,
                "article_id": aid,
                "gender": gender_lookup.get(aid, "unknown"),
                "rank_before": retrieved_rank[aid],
                "rank_after": reranked_rank.get(aid,-20),
                "rank_shift": retrieved_rank[aid] - reranked_rank.get(aid,-20),  # positive = moved up (LLM ranked it higher)
            })
    return pd.DataFrame(rows)

comparison_df = build_rank_comparison(retrieved, parsed_results_dedupe, gender_map)

In [236]:
# filter rows where article_id doesn't exist
comparison_df = comparison_df[comparison_df['rank_after']!= -20] 
# comparison_df[comparison_df['rank_after']== -20].shape[0]/comparison_df.shape[0]
comparison_df.shape

(4153, 6)

In [237]:
summary = comparison_df.groupby("gender").agg(
    avg_rank_before=("rank_before", "mean"),
    avg_rank_after=("rank_after", "mean"),
    avg_shift=("rank_shift", "mean"),
    n=("article_id", "count"),
)
print(summary)

         avg_rank_before  avg_rank_after  avg_shift     n
gender                                                   
F               5.172676        4.869070   0.303605   527
M               5.173318        5.078677   0.094641   877
Neutral         5.241906        5.249545  -0.007639  2749


In [238]:
from scipy.stats import mannwhitneyu

male_shifts = comparison_df.loc[comparison_df["gender"] == "M", "rank_shift"]
female_shifts = comparison_df.loc[comparison_df["gender"] == "F", "rank_shift"]

stat, p_value = mannwhitneyu(male_shifts, female_shifts, alternative="two-sided")
print(f"U-statistic: {stat}, p-value: {p_value}")

U-statistic: 216307.5, p-value: 0.037321934683875894


In [239]:
print(comparison_df["gender"].value_counts())

gender
Neutral    2749
M           877
F           527
Name: count, dtype: int64


In [241]:
n1, n2 = len(male_shifts), len(female_shifts)
u_stat = 216307.5  # your reported U
rank_biserial = 1 - (2 * u_stat) / (n1 * n2)
print(f"Rank-biserial correlation: {rank_biserial:.4f}")

Rank-biserial correlation: 0.0640


In [242]:
overall_weighted_shift = (
    summary["avg_shift"] * summary["n"]
).sum() / summary["n"].sum()
print(overall_weighted_shift)

0.05345533349385986


In [222]:
retrieved_dict = retrieved
reranked_dict = parsed_results_dedupe

In [243]:
def diagnose_reranking(retrieved_ids, reranked_ids):
    retrieved_set = set(retrieved_ids)
    reranked_set = set(reranked_ids)
    
    hallucinated = reranked_set - retrieved_set
    missing = retrieved_set - reranked_set
    duplicates = len(reranked_ids) - len(reranked_set)
    
    return {
        "n_hallucinated": len(hallucinated),
        "n_missing": len(missing),
        "n_duplicates": duplicates,
        "is_clean": len(hallucinated) == 0 and len(missing) == 0 and duplicates == 0,
    }

diagnostics = []
for user_id in reranked_dict:  # iterate over the SAMPLE, not the full retrieved set
    d = diagnose_reranking(retrieved_dict[user_id], reranked_dict[user_id])
    d["user_id"] = user_id
    diagnostics.append(d)

diag_df = pd.DataFrame(diagnostics)

print(f"Sample size: {len(diag_df)}")
print(f"Clean rerankings: {diag_df['is_clean'].sum()} / {len(diag_df)}")
print(f"Users with hallucinated IDs: {(diag_df['n_hallucinated'] > 0).sum()}")
print(f"Users with missing IDs: {(diag_df['n_missing'] > 0).sum()}")
print(f"Users with duplicates: {(diag_df['n_duplicates'] > 0).sum()}")

Sample size: 494
Clean rerankings: 111 / 494
Users with hallucinated IDs: 357
Users with missing IDs: 383
Users with duplicates: 0


In [244]:
diag_df

,n_hallucinated,n_missing,n_duplicates,is_clean,user_id
0,1,1,0,False,U69281
1,2,2,0,False,U92134
2,0,0,0,True,U72878
3,3,3,0,False,U17094
4,2,2,0,False,U37912
...,...,...,...,...,...
489,1,1,0,False,U2545
490,1,1,0,False,U4150
491,1,1,0,False,U7511
492,2,2,0,False,U56042


In [251]:
# set(reranked_dict['U69281']) - set(retrieved_dict['U69281'])
# N28620 {'N34520'}
retrieved_dict['U69281']

['N10406',
 'N39316',
 'N29542',
 'N41988',
 'N60397',
 'N40023',
 'N19810',
 'N30622',
 'N35738',
 'N28620']

In [252]:
length_check = pd.DataFrame([
    {"user_id": u, "retrieved_len": len(retrieved_dict[u]), "reranked_len": len(reranked_dict[u])}
    for u in reranked_dict
])

print(length_check[["retrieved_len", "reranked_len"]].describe())
print((length_check["retrieved_len"] != length_check["reranked_len"]).sum(), "users have mismatched list lengths")

       retrieved_len  reranked_len
count          494.0    494.000000
mean            10.0      9.763158
std              0.0      0.943956
min             10.0      1.000000
25%             10.0     10.000000
50%             10.0     10.000000
75%             10.0     10.000000
max             10.0     10.000000
58 users have mismatched list lengths


In [253]:
length_check = pd.DataFrame([
    {"user_id": u, "reranked_len": len(reranked_dict[u])}
    for u in reranked_dict
]).sort_values("reranked_len", ascending=False)

print(length_check.head(10))

# look at the raw text for the worst one
worst_user = length_check.iloc[0]["user_id"]
for k, v in results.items():
    if k.split(':')[0]==worst_user:
        print(repr(results[k])[:1000])  # however your raw pre-parsed responses are stored

    user_id  reranked_len
0    U69281            10
312  U37672            10
327  U80889            10
325  U49780            10
324  U29885            10
323  U27414            10
322   U2790            10
321  U87968            10
319  U13011            10
318  U46937            10
'[\n"N10406",\n"N39316",\n"N29542",\n"N41988",\n"N60397",\n"N40023",\n"N19810",\n"N30622",\n"N35738",\n"N34520"\n]'
'[\n"N10406",\n"N39316",\n"N29542",\n"N41988",\n"N60397",\n"N40023",\n"N19810",\n"N30622",\n"N35738",\n"N34520"\n]'
